# Church matching: ALIKED + LightGlue

A small experiment before we build a pipeline. We match a few reference churches (Onroerend Erfgoed, `kerken`) against a few thousand glass plates with local features, and measure two things:

1. **Speed** on this machine: time per image for feature extraction and time per pair for matching.
2. **Separation**: do true matches get clearly more RANSAC inliers than random pairs?

The plates are not random. For each church, we take the plates that DINOv2 finds most similar. That gives a good chance of true matches, and the pairs between a church and the plates of *other* churches act as negatives.

## Parameters

In [ ]:
from pathlib import Path

from common.config import config

# the DINOv2 stores and the church images sit next to the configured plate data
PLATE_DIR = Path(config().paths.images.glashelder)
CHURCH_DIR = PLATE_DIR.parent / "onroerend" / "kerken"
EMBEDDINGS = Path(config().paths.embeddings.glashelder).parent
PLATE_STORE = EMBEDDINGS / "glashelder-dino.store"
CHURCH_STORE = EMBEDDINGS / "onroerend-dino.store"

N_CHURCHES = 20
PLATES_PER_CHURCH = 100  # DINOv2 neighbours per church; their union is the plate set
SEED = 0

RESIZE = 1024  # long side for extraction
MAX_KEYPOINTS = 2048
RANSAC_PX = 3.0  # at RESIZE scale

PLATE_DIR, CHURCH_DIR, PLATE_STORE, CHURCH_STORE

## Setup

Check that the GPU is seen. On the DGX Spark this must say `cuda` and name the GB10.

In [ ]:
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from lightglue import ALIKED, LightGlue, viz2d
from lightglue.utils import numpy_image_to_torch, rbd
from PIL import Image
from tqdm.auto import tqdm

from common.images import load
from embed.device import get_device
from embed.io import load_embeddings
from embed.search import place, topk

device = get_device()
print(
    torch.__version__, device, torch.cuda.get_device_name() if device == "cuda" else ""
)


def sync():
    if device == "cuda":
        torch.cuda.synchronize()


def gray(ref) -> Image.Image:
    """Grayscale as 3 channels, so the colour references look like the plates."""
    return load(ref).convert("L").convert("RGB")


def column(rows: list[dict], key: str) -> np.ndarray:
    return np.array([r[key] for r in rows])

## Sample

Pick random churches from the church store, then the plates closest to each of them in the plate store. Both stores are `dinov2-large`, so their vectors compare directly.

In [ ]:
church_store = load_embeddings(CHURCH_STORE, root=CHURCH_DIR)
plate_store = load_embeddings(PLATE_STORE, root=PLATE_DIR)
assert church_store["model_id"] == plate_store["model_id"], (
    "stores use different models"
)

rows = random.Random(SEED).sample(
    range(len(church_store["images"])),
    min(N_CHURCHES, len(church_store["images"])),
)
churches = [church_store["images"][i] for i in rows]
print(f"{len(churches)} of {len(church_store['images'])} churches")

plate_embeddings = place(plate_store["embeddings"], device, label="plates")
dino_sim, dino_idx = topk(
    church_store["embeddings"][rows], plate_embeddings, PLATES_PER_CHURCH
)
plates = sorted({plate_store["images"][i] for i in dino_idx.flatten().tolist()})
dino_rank = {
    (church, plate_store["images"][i]): rank
    for church, row in zip(churches, dino_idx.tolist(), strict=True)
    for rank, i in enumerate(row)
}
print(f"{len(plates)} plates, {len(churches) * len(plates):,} pairs")

del plate_embeddings
if device == "cuda":
    torch.cuda.empty_cache()

## Extract features

Once per image. Keypoints are kept at `RESIZE` scale for RANSAC, so the pixel threshold means the same for plates and churches. Image reading is timed apart from the model.

In [ ]:
extractor = ALIKED(max_num_keypoints=MAX_KEYPOINTS).eval().to(device)
matcher = LightGlue(features="aliked").eval().to(device)


def extract(ref) -> tuple[dict, np.ndarray, float, float]:
    start = time.perf_counter()
    image = numpy_image_to_torch(np.asarray(gray(ref))).to(device)
    read = time.perf_counter()
    with torch.inference_mode():
        feats = extractor.extract(image, resize=RESIZE)
    sync()
    done = time.perf_counter()
    scale = RESIZE / max(image.shape[-2:])
    keypoints = feats["keypoints"][0].cpu().numpy() * scale
    return feats, keypoints, read - start, done - read


extract(churches[0])  # warm up

refs = churches + plates
extracted = {ref: extract(ref) for ref in tqdm(refs, desc="extract")}
feats = {ref: e[0] for ref, e in extracted.items()}
keypoints = {ref: e[1] for ref, e in extracted.items()}
read_s = np.array([e[2] for e in extracted.values()])
model_s = np.array([e[3] for e in extracted.values()])
n_kpts = np.array([len(k) for k in keypoints.values()])

print(f"read:    {read_s.mean() * 1e3:6.1f} ms/image")
print(f"extract: {model_s.mean() * 1e3:6.1f} ms/image")
print(f"keypoints: median {np.median(n_kpts):.0f}, min {n_kpts.min()}")

## Match

Every church against every plate in the sample. A fundamental matrix (MAGSAC) checks the matches, since a facade seen from another position is not a plane. The score is the inlier count.

In [ ]:
def inliers(pts0: np.ndarray, pts1: np.ndarray) -> np.ndarray:
    if len(pts0) < 8:
        return np.zeros(len(pts0), dtype=bool)
    _, mask = cv2.findFundamentalMat(
        pts0, pts1, cv2.USAC_MAGSAC, RANSAC_PX, 0.999, 10_000
    )
    return (
        np.zeros(len(pts0), dtype=bool) if mask is None else mask.ravel().astype(bool)
    )


def lightglue(church: str, plate: str) -> np.ndarray:
    with torch.inference_mode():
        out = rbd(matcher({"image0": feats[church], "image1": feats[plate]}))
    return out["matches"].cpu().numpy()  # waits for the GPU


def match(church: str, plate: str) -> dict:
    start = time.perf_counter()
    pairs = lightglue(church, plate)
    matched = time.perf_counter()
    mask = inliers(keypoints[church][pairs[:, 0]], keypoints[plate][pairs[:, 1]])
    return {
        "church": church,
        "plate": plate,
        "matches": len(pairs),
        "inliers": int(mask.sum()),
        "dino_rank": dino_rank.get((church, plate)),
        "match_s": matched - start,
        "ransac_s": time.perf_counter() - matched,
    }


match(churches[0], plates[0])  # warm up

pairs = [(c, p) for c in churches for p in plates]
start = time.perf_counter()
results = [match(c, p) for c, p in tqdm(pairs, desc="match")]
wall = time.perf_counter() - start

print(f"lightglue: {column(results, 'match_s').mean() * 1e3:6.1f} ms/pair")
print(f"ransac:    {column(results, 'ransac_s').mean() * 1e3:6.1f} ms/pair")
print(f"wall:      {wall / len(pairs) * 1e3:6.1f} ms/pair over {len(pairs):,} pairs")

## What would the full run cost?

One pair at a time, as above. Batching LightGlue should make this faster, so take it as an upper bound.

In [ ]:
per_pair = wall / len(pairs)
per_image = (read_s + model_s).mean()
n_plates, n_churches = len(plate_store["images"]), len(church_store["images"])


def hours(seconds: float) -> str:
    return f"{seconds / 3600:8.1f} h"


print("extract all plates and churches:", hours((n_plates + n_churches) * per_image))
print("all pairs:                      ", hours(n_plates * n_churches * per_pair))
print("shortlist, ~300k pairs:         ", hours(300_000 * per_pair))

## Inlier distribution

Most pairs are not the same church. If the method works, they pile up near zero, and the true matches form a separate tail on the right.

In [ ]:
counts = column(results, "inliers")
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(counts, bins=np.arange(0, counts.max() + 5, 2))
ax.set_yscale("log")
ax.set_xlabel("RANSAC inliers")
ax.set_ylabel("pairs")
plt.show()


def table(rows: list[dict]):
    print(f"{'church':>12} {'plate':>40} {'matches':>8} {'inliers':>8} {'dino':>5}")
    for r in rows:
        rank = "" if r["dino_rank"] is None else r["dino_rank"]
        print(
            f"{Path(r['church']).name:>12} {Path(r['plate']).name:>40} {r['matches']:>8} {r['inliers']:>8} {rank:>5}"
        )


best = sorted(results, key=lambda r: r["inliers"], reverse=True)
table(best[:30])

## Look at the best pairs

Only inliers are drawn. Check by eye which ones are really the same church, and note the inlier count where false matches start.

In [ ]:
def size(ref: str) -> tuple[int, int]:
    w, h = feats[ref]["image_size"][0].tolist()
    scale = RESIZE / max(w, h)
    return round(w * scale), round(h * scale)


def show(church: str, plate: str):
    pairs = lightglue(church, plate)
    pts0, pts1 = keypoints[church][pairs[:, 0]], keypoints[plate][pairs[:, 1]]
    mask = inliers(pts0, pts1)
    images = [np.asarray(gray(ref).resize(size(ref))) for ref in (church, plate)]
    viz2d.plot_images(
        images,
        titles=[Path(church).name, f"{Path(plate).name}  ({mask.sum()} inliers)"],
    )
    viz2d.plot_matches(pts0[mask], pts1[mask], lw=0.5)
    plt.show()


for row in best[:12]:
    show(row["church"], row["plate"])

## Would DINOv2 have put them in the shortlist?

`dino` is where the plate stood among the DINOv2 neighbours of that church (empty: it came in through another church). After you have checked which pairs are real, set the threshold below and see how far down the DINOv2 list they were. If true matches are often missing (empty rank), DINOv2 is not good enough as a shortlist, and we should try MegaLoc.

In [ ]:
THRESHOLD = 50  # set this from the plots above

found = [r for r in best if r["inliers"] >= THRESHOLD]
missed = sum(r["dino_rank"] is None for r in found)
print(f"{len(found)} pairs with >= {THRESHOLD} inliers")
print(f"{missed} of them not in the DINOv2 top {PLATES_PER_CHURCH} of their church")
table(found)